# ROC Curves — All Models

Models compared:
- **Paper baseline** (`EMBER2024_all.model` — LightGBM, all file types)
- **Our 64-leaf LightGBM** (trained on full training set)
- **Best ensemble** (per-type × 0.70 + paper_all × 0.20 + 64-leaf × 0.10)
- **Neural model** (ResidualMLP + SupCon, smoke-test subset, 2 000 samples)

Scores for LightGBM models are cached in `results/roc_scores_lgbm.npz` after the first run.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score

sys.path.insert(0, str(Path.cwd().parent))

matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

RESULTS_DIR  = Path('../results')
CKPT_DIR     = Path('../checkpoints')
ARTIFACTS    = Path('/Users/roheeeee/Documents/DACS/EMBER2024-artifacts')
DATA_DIR     = Path('/Users/roheeeee/Documents/DACS/EMBER2024-corrected-full')
CH_JSONL     = Path('/Users/roheeeee/Documents/DACS/EMBER2024-full-local/2023-09-24_2024-12-14_challenge_malicious.jsonl')

CACHE_PATH   = RESULTS_DIR / 'roc_scores_lgbm.npz'

FT_MODEL = {
    'Win32':   'EMBER2024_Win32.model',
    'Win64':   'EMBER2024_Win64.model',
    'Dot_Net': 'EMBER2024_Dot_Net.model',
    'APK':     'EMBER2024_APK.model',
    'ELF':     'EMBER2024_ELF.model',
    'PDF':     'EMBER2024_PDF.model',
}
WEIGHTS = dict(per_type=0.70, paper_all=0.20, our64=0.10)
BATCH = 50_000

## 1. Generate / Load LightGBM Scores

In [ ]:
def predict_batched(booster, X, indices=None):
    if indices is None:
        indices = np.arange(len(X))
    out = []
    for i in range(0, len(indices), BATCH):
        batch = indices[i : i + BATCH]
        out.append(booster.predict(np.array(X[batch])))
    return np.concatenate(out)

def score_per_type(X, ftypes, boosters):
    scores = np.zeros(len(X))
    for ft, b in boosters.items():
        mask = ftypes == ft
        if not mask.any():
            continue
        scores[mask] = predict_batched(b, X, np.where(mask)[0])
    return scores

if CACHE_PATH.exists():
    print(f'Loading cached scores from {CACHE_PATH}')
    cache = np.load(CACHE_PATH, allow_pickle=True)
    y_test          = cache['y_test']
    score_paper_all = cache['score_paper_all']
    score_64leaf    = cache['score_64leaf']
    score_per_type_ = cache['score_per_type']
    score_ensemble  = cache['score_ensemble']
    y_ch_combined   = cache['y_ch_combined']
    ch_paper_all    = cache['ch_paper_all']
    ch_64leaf       = cache['ch_64leaf']
    ch_per_type_    = cache['ch_per_type']
    ch_ensemble     = cache['ch_ensemble']
    print(f'  Test samples:      {len(y_test):,}')
    print(f'  Challenge samples: {len(y_ch_combined):,}')
else:
    import lightgbm as lgb

    print('Loading models...')
    boosters_pt = {ft: lgb.Booster(model_file=str(ARTIFACTS / mn)) for ft, mn in FT_MODEL.items()}
    booster_all = lgb.Booster(model_file=str(ARTIFACTS / 'EMBER2024_all.model'))
    booster_64  = lgb.Booster(model_file=str(CKPT_DIR / 'lgbm/lgbm_64leaf.txt'))

    print('Loading test data...')
    X_test = np.memmap(DATA_DIR / 'X_test.dat', dtype=np.float32, mode='r', shape=(605929, 2568))
    y_test = np.array(np.memmap(DATA_DIR / 'y_test.dat', dtype=np.int32, mode='r', shape=(605929,)))

    test_ftypes = np.array([json.loads(l).get('file_type', 'unknown')
                            for l in open(DATA_DIR / '2024-09-22_2024-12-14_test.jsonl')])

    print('Scoring test set (paper_all)...')
    score_paper_all = predict_batched(booster_all, X_test)
    print('Scoring test set (64leaf)...')
    score_64leaf    = predict_batched(booster_64,  X_test)
    print('Scoring test set (per-type)...')
    score_per_type_ = score_per_type(X_test, test_ftypes, boosters_pt)
    score_ensemble  = (WEIGHTS['per_type']  * score_per_type_ +
                       WEIGHTS['paper_all'] * score_paper_all +
                       WEIGHTS['our64']     * score_64leaf)

    print('Loading challenge data...')
    X_ch = np.memmap(DATA_DIR / 'X_challenge.dat', dtype=np.float32, mode='r', shape=(6315, 2568))
    ch_ftypes = np.array([json.loads(l).get('file_type', 'unknown') for l in open(CH_JSONL)])

    # Challenge ROC: malicious challenge samples vs benign test samples
    ben_idx = np.flatnonzero(y_test == 0)

    print('Scoring challenge set...')
    ch_paper_all_raw  = predict_batched(booster_all, X_ch)
    ch_64leaf_raw     = predict_batched(booster_64,  X_ch)
    ch_per_type_raw   = score_per_type(X_ch, ch_ftypes, boosters_pt)
    ch_ensemble_raw   = (WEIGHTS['per_type']  * ch_per_type_raw +
                         WEIGHTS['paper_all'] * ch_paper_all_raw +
                         WEIGHTS['our64']     * ch_64leaf_raw)

    # Combine: benign from test + all challenge malicious
    n_ben = len(ben_idx)
    y_ch_combined  = np.concatenate([np.zeros(n_ben), np.ones(6315)])
    ch_paper_all   = np.concatenate([score_paper_all[ben_idx], ch_paper_all_raw])
    ch_64leaf      = np.concatenate([score_64leaf[ben_idx],    ch_64leaf_raw])
    ch_per_type_   = np.concatenate([score_per_type_[ben_idx], ch_per_type_raw])
    ch_ensemble    = np.concatenate([score_ensemble[ben_idx],  ch_ensemble_raw])

    np.savez_compressed(
        CACHE_PATH,
        y_test=y_test,
        score_paper_all=score_paper_all,
        score_64leaf=score_64leaf,
        score_per_type=score_per_type_,
        score_ensemble=score_ensemble,
        y_ch_combined=y_ch_combined,
        ch_paper_all=ch_paper_all,
        ch_64leaf=ch_64leaf,
        ch_per_type=ch_per_type_,
        ch_ensemble=ch_ensemble,
    )
    print(f'Cached → {CACHE_PATH}')

## 2. Load Neural Model Smoke Scores

In [ ]:
smoke_path = RESULTS_DIR / 'smoke/raw_scores.npz'
neural_test_y     = None
neural_test_score = None
neural_ch_y       = None
neural_ch_score   = None

if smoke_path.exists():
    smoke = np.load(smoke_path, allow_pickle=True)
    neural_test_y     = smoke['test_y_true']
    neural_test_score = smoke['test_our_score']
    neural_ch_y       = smoke['challenge_y_true']
    neural_ch_score   = smoke['challenge_our_score']
    print(f'Neural test   samples: {len(neural_test_y):,} (smoke subset)')
    print(f'Neural challenge samples: {len(neural_ch_y):,} (smoke subset)')
else:
    print('Smoke scores not found — neural model will be omitted from plots.')

## 3. ROC Curves — Test Set

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

models_test = [
    ('Best Ensemble',      score_ensemble,  y_test, '#1f77b4', '-'),
    ('Paper Baseline (all)', score_paper_all, y_test, '#ff7f0e', '--'),
    ('Our 64-leaf LGBM',   score_64leaf,    y_test, '#2ca02c', '-.'),
]

for label, scores, labels, color, ls in models_test:
    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, linestyle=ls,
            label=f'{label} (AUC = {roc_auc:.4f})')

if neural_test_y is not None and len(np.unique(neural_test_y)) > 1:
    fpr_n, tpr_n, _ = roc_curve(neural_test_y, neural_test_score)
    roc_n = auc(fpr_n, tpr_n)
    ax.plot(fpr_n, tpr_n, color='#d62728', lw=2, linestyle=':',
            label=f'Neural (smoke, n=2k) (AUC = {roc_n:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.01)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_test.pdf', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'roc_test.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved roc_test.pdf / .png')

## 4. ROC Curves — Challenge Set

The challenge set contains only evasive malware samples. Following the paper's evaluation protocol, the ROC is computed against benign samples from the test set.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

models_ch = [
    ('Best Ensemble',        ch_ensemble,  y_ch_combined, '#1f77b4', '-'),
    ('Paper Baseline (all)', ch_paper_all, y_ch_combined, '#ff7f0e', '--'),
    ('Our 64-leaf LGBM',     ch_64leaf,    y_ch_combined, '#2ca02c', '-.'),
]

for label, scores, labels, color, ls in models_ch:
    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, linestyle=ls,
            label=f'{label} (AUC = {roc_auc:.4f})')

if neural_ch_y is not None and len(np.unique(neural_ch_y)) > 1:
    fpr_n, tpr_n, _ = roc_curve(neural_ch_y, neural_ch_score)
    roc_n = auc(fpr_n, tpr_n)
    ax.plot(fpr_n, tpr_n, color='#d62728', lw=2, linestyle=':',
            label=f'Neural (smoke, n=100) (AUC = {roc_n:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.01)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Challenge Set (evasive malware)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_challenge.pdf', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'roc_challenge.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved roc_challenge.pdf / .png')

## 5. Combined Figure (Test + Challenge side-by-side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

palette = {
    'Best Ensemble':          ('#1f77b4', '-'),
    'Paper Baseline (all)':   ('#ff7f0e', '--'),
    'Our 64-leaf LGBM':       ('#2ca02c', '-.'),
    'Neural (smoke)':         ('#d62728', ':'),
}

for ax, (split_label, models) in zip(axes, [
    ('Test Set', [
        ('Best Ensemble',        score_ensemble,  y_test),
        ('Paper Baseline (all)', score_paper_all, y_test),
        ('Our 64-leaf LGBM',     score_64leaf,    y_test),
    ] + ([('Neural (smoke)', neural_test_score, neural_test_y)]
         if neural_test_y is not None and len(np.unique(neural_test_y)) > 1 else [])),
    ('Challenge Set', [
        ('Best Ensemble',        ch_ensemble,  y_ch_combined),
        ('Paper Baseline (all)', ch_paper_all, y_ch_combined),
        ('Our 64-leaf LGBM',     ch_64leaf,    y_ch_combined),
    ] + ([('Neural (smoke)', neural_ch_score, neural_ch_y)]
         if neural_ch_y is not None and len(np.unique(neural_ch_y)) > 1 else [])),
]):
    for name, scores, labels in models:
        color, ls = palette[name]
        fpr, tpr, _ = roc_curve(labels, scores)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, linestyle=ls,
                label=f'{name} ({roc_auc:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC — {split_label}')
    ax.legend(loc='lower right', title='Model (AUC)')
    ax.grid(True, alpha=0.3)

plt.suptitle('EMBER2024 Malware Detection — ROC Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_combined.pdf', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'roc_combined.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved roc_combined.pdf / .png')

## 6. Low-FPR Zoom (Test Set)

Security-relevant operating region: FPR ≤ 1%

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for label, scores, labels, color, ls in models_test:
    fpr, tpr, _ = roc_curve(labels, scores)
    mask = fpr <= 0.01
    ax.plot(fpr[mask], tpr[mask], color=color, lw=2, linestyle=ls, label=label)

if neural_test_y is not None and len(np.unique(neural_test_y)) > 1:
    fpr_n, tpr_n, _ = roc_curve(neural_test_y, neural_test_score)
    mask_n = fpr_n <= 0.01
    ax.plot(fpr_n[mask_n], tpr_n[mask_n], color='#d62728', lw=2, linestyle=':', label='Neural (smoke, n=2k)')

ax.set_xlim(0, 0.01)
ax.set_ylim(0, 1.01)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set (zoomed: FPR ≤ 1%)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_test_zoom.pdf', bbox_inches='tight')
plt.savefig(RESULTS_DIR / 'roc_test_zoom.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved roc_test_zoom.pdf / .png')